Reproducing Nakamoto's double-spending probability
This notebook reproduces the attacker's success probability from Nakamoto's analysis, computes the minimum confirmation depth for a probability strictly below 0.1%, and compares Nakamoto with Rosenfeld on selected shared (q, z) cases.

The notebook expects theory.py (with attacker_success_probability and rosenfeld) to be importable from the project root.

In [2]:
from pathlib import Path
import pandas as pd
from theory import attacker_success_probability, rosenfeld

RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Nakamoto probability by confirmation depth
The table below reports the probability that the attacker eventually catches up and succeeds for selected attacker hash-power shares q and confirmation counts z.

In [3]:
q_values = [0.1, 0.2, 0.3, 0.4]
z_values = [1, 2, 3, 4, 5, 6, 10, 15, 20, 25, 50, 100]

nakamoto_rows = [
    {'q': q, 'z': z, 'success_probability': attacker_success_probability(q, z)}
    for q in q_values
    for z in z_values
]
nakamoto_table = pd.DataFrame(nakamoto_rows)
display(nakamoto_table.pivot(index='z', columns='q', values='success_probability'))
nakamoto_table.to_csv(RESULTS_DIR / 'nakamoto_probabilities.csv', index=False)

q,0.1,0.2,0.3,0.4
z,,,,
1,2.045873e-01,4.158994e-01,6.277491e-01,0.828861
2,5.097789e-02,2.039285e-01,4.457171e-01,0.736403
3,1.317224e-02,1.032416e-01,3.245841e-01,0.664168
4,3.455243e-03,5.299784e-02,2.391269e-01,0.603401
5,9.136822e-04,2.741554e-02,1.773523e-01,0.550625
6,2.428027e-04,1.425061e-02,1.321112e-01,0.503980
10,1.241402e-06,1.066953e-03,4.166048e-02,0.359976
15,1.739073e-09,4.287155e-05,1.010076e-02,0.241280
20,2.462997e-12,1.743361e-06,2.480398e-03,0.163560


Minimum number of confirmations for probability below 0.1%
For each q, this finds the smallest integer z for which the Nakamoto success probability is strictly less than 0.001 (0.1%).

In [4]:
threshold_rows = []
for q in q_values:
    z = 0
    probability = attacker_success_probability(q, z)
    while probability >= 0.001:
        z += 1
        probability = attacker_success_probability(q, z)
    threshold_rows.append({
        'q': q,
        'minimum_confirmations_z': z,
        'probability_at_z': probability,
        'probability_at_z_minus_1': attacker_success_probability(q, z - 1),
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.to_csv(RESULTS_DIR / 'nakamoto_z_below_0_1_percent.csv', index=False)

,q,minimum_confirmations_z,probability_at_z,probability_at_z_minus_1
0,0.1,5,0.000914,0.003455
1,0.2,11,0.000560,0.001067
2,0.3,24,0.000811,0.001072
3,0.4,89,0.000987,0.001061


In [5]:
threshold_rows = []
for q in q_values:
    z = 0
    probability = attacker_success_probability(q, z)
    while probability >= 0.001:
        z += 1
        probability = attacker_success_probability(q, z)
    threshold_rows.append({
        'q': q,
        'minimum_confirmations_z': z,
        'probability_at_z': probability,
        'probability_at_z_minus_1': attacker_success_probability(q, z - 1),
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.to_csv(RESULTS_DIR / 'nakamoto_z_below_0_1_percent.csv', index=False)

,q,minimum_confirmations_z,probability_at_z,probability_at_z_minus_1
0,0.1,5,0.000914,0.003455
1,0.2,11,0.000560,0.001067
2,0.3,24,0.000811,0.001072
3,0.4,89,0.000987,0.001061


Nakamoto vs. Rosenfeld
This comparison evaluates both expressions on the same selected (q, z) pairs. It is a numerical side-by-side comparison; the two formulas should not be assumed to have identical parameter interpretations without checking the definitions in the source papers.

In [6]:
comparison_cases = [
    (0.10, 1), (0.10, 2), (0.10, 5),
    (0.20, 3), (0.20, 6),
    (0.30, 4), (0.30, 10),
    (0.40, 5), (0.40, 10),
    (0.50, 3),
]

comparison_rows = []
for q, z in comparison_cases:
    comparison_rows.append({
        'q': q,
        'z': z,
        'Nakamoto': attacker_success_probability(q, z),
        'Rosenfeld': rosenfeld(q, z),
    })

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
comparison_table.to_csv(RESULTS_DIR / 'nakamoto_vs_rosenfeld.csv', index=False)

,q,z,Nakamoto,Rosenfeld
0,0.1,1,0.204587,0.200000
1,0.1,2,0.050978,0.056000
2,0.1,5,0.000914,0.001782
3,0.2,3,0.103242,0.115840
4,0.2,6,0.014251,0.023308
5,0.3,4,0.239127,0.252072
6,0.3,10,0.041660,0.065107
7,0.4,5,0.550625,0.533135
8,0.4,10,0.359976,0.372184
9,0.5,3,1.000000,1.000000


Reference-value checks
The following assertions check the two Nakamoto reference values used in test_theory.py.

In [7]:
assert abs(attacker_success_probability(0.1, 5) - 0.0009137) < 1e-7
assert abs(attacker_success_probability(0.3, 10) - 0.0416605) < 1e-7
print('Nakamoto reference values reproduced successfully.')

Nakamoto reference values reproduced successfully.
